# Kaggle: Predicción de precios de portátiles (Random Forest) 

- RAM: `'8GB' → 8`
- Peso: `'1.86kg' → 1.86`
- One-hot de categóricas sencillas: `Company`, `TypeName`, `OpSys`
- NO extrae CPU/GPU/Memory/Resolución.


In [18]:
import numpy as np
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Chequeador Kaggle (del notebook del profe)
import urllib.request
from PIL import Image

RANDOM_STATE = 42


## 1) Carga de datos

In [19]:
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')
sample = pd.read_csv('data/sample_submission.csv')

train.shape, test.shape, sample.shape

((912, 13), (391, 12), (391, 2))

## 2) Preprocesado SIMPLE 

In [20]:
train_p = train.copy()
test_p  = test.copy()

# RAM: '8GB' -> 8
for df in (train_p, test_p):
    df['Ram_GB'] = df['Ram'].str.extract(r'(\d+)').astype(float)

# Peso: '1.86kg' -> 1.86
for df in (train_p, test_p):
    df['Weight_kg'] = df['Weight'].str.lower().str.replace('kg','', regex=False)
    df['Weight_kg'] = pd.to_numeric(df['Weight_kg'], errors='coerce')

# Selección SIMPLE de features
keep_cols = ['laptop_ID', 'Inches', 'Ram_GB', 'Weight_kg', 'Company', 'TypeName', 'OpSys']
train_p = train_p[keep_cols + ['Price_in_euros']]
test_p  = test_p[keep_cols]

train_p.head()

,laptop_ID,Inches,Ram_GB,Weight_kg,Company,TypeName,OpSys,Price_in_euros
0,755,15.6,8.0,1.86,HP,Notebook,Windows 10,539.00
1,618,15.6,16.0,2.59,Dell,Gaming,Windows 10,879.01
2,909,15.6,8.0,2.04,HP,Notebook,Windows 10,900.00
3,2,13.3,8.0,1.34,Apple,Ultrabook,macOS,898.94
4,286,15.6,4.0,2.25,Dell,Notebook,Linux,428.00


## 3) Nulos + One-Hot + alineación

In [21]:
y = train_p['Price_in_euros']
X = train_p.drop(columns=['Price_in_euros'])

num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object','bool']).columns.tolist()

# imputación numéricas
med = X[num_cols].median()
X[num_cols] = X[num_cols].fillna(med)
test_p[num_cols] = test_p[num_cols].fillna(med)

# imputación categóricas
for c in cat_cols:
    mode_val = X[c].mode(dropna=True)[0]
    X[c] = X[c].fillna(mode_val)
    test_p[c] = test_p[c].fillna(mode_val)

# Guardar IDs fuera del modelo
id_test  = test_p['laptop_ID'].copy()
X = X.drop(columns=['laptop_ID'])
test_feats = test_p.drop(columns=['laptop_ID'])

# One-hot + alinear
X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=True)
test_enc = pd.get_dummies(test_feats, columns=cat_cols, drop_first=True)
test_enc = test_enc.reindex(columns=X_enc.columns, fill_value=0)

X_enc.shape, test_enc.shape

C:\Users\carlo\AppData\Local\Temp\ipykernel_24768\3179834382.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_p[num_cols] = test_p[num_cols].fillna(med)
C:\Users\carlo\AppData\Local\Temp\ipykernel_24768\3179834382.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_p[c] = test_p[c].fillna(mode_val)


((912, 34), (391, 34))

## 4) Validación local (RMSE)

In [22]:
X_train, X_val, y_train, y_val = train_test_split(
    X_enc, y, test_size=0.2, random_state=RANDOM_STATE
)

rf = RandomForestRegressor(
    n_estimators=600,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred_train = rf.predict(X_train)
pred_val   = rf.predict(X_val)

rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
rmse_val   = np.sqrt(mean_squared_error(y_val, pred_val))

print(f"RMSE train: {rmse_train:.3f}")
print(f"RMSE val:   {rmse_val:.3f}")
print(f"Diferencia: {rmse_val - rmse_train:.3f}")

RMSE train: 152.139
RMSE val:   388.803
Diferencia: 236.663


## 5) Entrenamiento final + submission

In [14]:
rf_final = RandomForestRegressor(
    n_estimators=800,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_final.fit(X_enc, y)

test_preds = rf_final.predict(test_enc)

submission = pd.DataFrame({
    'laptop_ID': id_test.values,
    'Price_in_euros': test_preds
})

submission.head(), submission.shape

(   laptop_ID  Price_in_euros
 0        209     1404.975664
 1       1281      349.218847
 2       1168      359.716916
 3       1231      876.993952
 4       1020     1135.848850,
 (391, 2))

## 6) Chequeador Kaggle

In [15]:
def chequeador(df_to_submit):
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index=False)
                urllib.request.urlretrieve(
                    "https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg",
                    "gfg.png"
                )
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: has tocado filas de test.csv :(")


In [16]:
chequeador(submission)

You're ready to submit!
